# BÀI TẬP LỚN HỌC PHẦN: NHẬP MÔN HỌC MÁY (FIT4013)
## Đề tài 8: Dự đoán sinh viên có nguy cơ bỏ học (Nhóm 9)
- **Giảng viên hướng dẫn:** ThS. Lê Thị Thùy Trang
- **Thành viên thực hiện:**
  1. Nguyễn Hữu Tiến (`1871020569` - CNTT 18-10)
  2. Vương Xuân Sơn (`1871020502` - CNTT 18-10)
  3. Lý Triệu Trung (`1871020698` - CNTT 18-10)

Notebook thực hiện quy trình Học máy khép kín (End-to-End): **Nguồn dữ liệu -> EDA -> Tiền xử lý & Kỹ thuật đặc trưng (Luồng A vs. Luồng B) -> Huấn luyện & Tinh chỉnh tham số -> Đánh giá đa chỉ số -> Phân tích lỗi & Phân tầng rủi ro**.

In [ ]:
# 1. Thiết lập đường dẫn gốc dự án để gọi các module trong thư mục src/
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from IPython.display import Image, display

from src.utils import (
    DATA_PATH, FIGURES_DIR, RESULTS_CSV_PATH, TUNING_CSV_PATH,
    ERROR_CSV_PATH, BEST_MODEL_PATH, set_seed, ensure_directories
)
from src.preprocessing import (
    load_and_clean_data, print_eda_overview, get_outlier_summary_iqr,
    split_train_test, build_preprocessor_flow_a, build_preprocessor_flow_b
)
from src.train_evaluate import run_experiment_pipeline
from run_pipeline import ensure_all_figures_exist

set_seed(42)
ensure_directories()
print('Đã nạp thành công các module từ src/ ! Thư mục gốc:', PROJECT_ROOT)

## Bước 1 & 2: Đọc dữ liệu và Khám phá dữ liệu (EDA)
- Kiểm tra kích thước (`4.424 dòng x 37 cột`), dữ liệu thiếu (`null = 0`), trùng lặp (`duplicates = 0`) và phân phối nhãn mục tiêu.
- Thống kê mô tả, phát hiện ngoại lệ theo quy tắc $1.5 \times \text{IQR}$ và tính hệ số tương quan Pearson với nhãn Bỏ học.

In [ ]:
df = load_and_clean_data(DATA_PATH)
print_eda_overview(df)
df.head(5)

In [ ]:
# Thống kê mô tả, số lượng ngoại lệ (1.5 * IQR) và tương quan với nhãn Bỏ học
key_numeric_cols = [
    'Previous qualification (grade)', 'Admission grade', 'Age at enrollment',
    'Curricular units 1st sem (enrolled)', 'Curricular units 1st sem (approved)', 'Curricular units 1st sem (grade)',
    'Curricular units 2nd sem (enrolled)', 'Curricular units 2nd sem (approved)', 'Curricular units 2nd sem (grade)',
    'Unemployment rate', 'Inflation rate', 'GDP'
]
eda_stats_df = get_outlier_summary_iqr(df, key_numeric_cols)
eda_stats_df

## Bước 3: Chia dữ liệu Stratified (80/20) và Thiết kế Luồng A vs. Luồng B
- **Luồng A (Pipeline cơ bản):** Giữ nguyên 36 thuộc tính gốc, chuẩn hóa `StandardScaler`.
- **Luồng B (Pipeline cải tiến):** Tạo 7 đặc trưng miền giáo dục (`Approval_rate_1st_sem`, `Approval_rate_2nd_sem`, `Grade_diff_2nd_1st`, `Avg_grade_both_sem`, `Total_approved_units`, `Total_without_eval`, `Financial_risk_score`) + Mã hóa `OneHotEncoder(min_frequency=0.01)` cho 9 biến định danh + `StandardScaler`.

In [ ]:
X_train, X_test, y_train, y_test = split_train_test(df, test_size=0.20)
preproc_a = build_preprocessor_flow_a()
preproc_b = build_preprocessor_flow_b(list(X_train.columns))

XA_train = preproc_a.fit_transform(X_train)
XB_train = preproc_b.fit_transform(X_train)

print(f'Kích thước tập Train : {X_train.shape[0]} mẫu (Dropout = {y_train.sum()})')
print(f'Kích thước tập Test  : {X_test.shape[0]} mẫu (Dropout = {y_test.sum()})')
print(f'Số chiều đặc trưng sau Luồng A (Cơ bản)  : {XA_train.shape[1]} chiều')
print(f'Số chiều đặc trưng sau Luồng B (Cải tiến): {XB_train.shape[1]} chiều')

## Bước 4 & 5: Huấn luyện mô hình, Khảo sát siêu tham số và Đánh giá đa chỉ số
- So sánh 3 thuật toán bắt buộc (**K-NN**, **Naive Bayes**, **Decision Tree**) và thuật toán mở rộng (**Random Forest**) trên cả Luồng A và Luồng B.
- Đánh giá bằng 6 độ đo: `Accuracy`, `Precision`, `Recall`, `F1-score`, `ROC-AUC`, `PR-AUC`.

In [ ]:
pipeline_outputs = run_experiment_pipeline()
if len(pipeline_outputs) == 3:
    results_df, tuning_df, error_df = pipeline_outputs
else:
    results_df, tuning_df = pipeline_outputs
    error_df = pd.read_csv(ERROR_CSV_PATH)

ensure_all_figures_exist(df)
results_df

In [ ]:
# Bảng kết quả khảo sát siêu tham số (Hyperparameter Tuning)
tuning_df

In [ ]:
# Hiển thị toàn bộ 7 biểu đồ trực quan (EDA & Đánh giá mô hình)
for img_name in [
    'hinh1_phan_phoi_nhan.png',
    'hinh2_tai_chinh_tuong_quan.png',
    'hinh3_boxplot_hoc_tap.png',
    'hinh4_so_sanh_luong_A_B.png',
    'hinh5_roc_auc_curve.png',
    'hinh6_confusion_matrix.png',
    'hinh7_feature_importance.png'
]:
    img_path = FIGURES_DIR / img_name
    if img_path.exists():
        display(Image(filename=str(img_path), width=760))

## Bước 6: Phân tích lỗi (Error Analysis) và Phân tầng rủi ro 3 mức
- Phân tích các ca dự đoán sai trên tập kiểm thử: **Âm tính giả (False Negative - FN)** và **Dương tính giả (False Positive - FP)**.
- Phân tầng sinh viên thành 3 mức rủi ro: **Thấp (< 40%)**, **Trung bình (40% - 70%)**, **Cao (>= 70%)**.

In [ ]:
err_col = 'Error_Type' if 'Error_Type' in error_df.columns else 'Prediction_Status'
print('=== THỐNG KÊ TRẠNG THÁI DỰ ĐOÁN TRÊN TẬP KIỂM THỬ (N = 885) ===')
print(error_df[err_col].value_counts().to_string())

tier_table = error_df.groupby('Risk_Tier')['Actual_Dropout'].agg(
    So_luong_SV='count', So_bo_hoc_that='sum', Ty_le_bo_hoc_that='mean'
).reset_index()
tier_table['Ty_le_bo_hoc_that'] = (tier_table['Ty_le_bo_hoc_that'] * 100).round(2)
tier_table

In [ ]:
# 5 trường hợp Âm tính giả (FN - Sinh viên thực tế bỏ học nhưng mô hình dự đoán Không bỏ học)
cols_inspect = [
    c for c in [
        'Test_Student_Index', 'Actual_Dropout', 'Predicted_Dropout', 'Dropout_Probability', 'Risk_Tier',
        'Curricular units 1st sem (approved)', 'Curricular units 2nd sem (approved)',
        'Curricular units 2nd sem (grade)', 'Tuition fees up to date', 'Debtor', 'Scholarship holder'
    ] if c in error_df.columns
]
error_df[error_df[err_col].str.contains('FN|False Negative')][cols_inspect].head(5)

In [ ]:
# 5 trường hợp Dương tính giả (FP - Sinh viên không bỏ học nhưng mô hình cảnh báo nhầm)
error_df[error_df[err_col].str.contains('FP|False Positive')][cols_inspect].head(5)